# 5장. 분석을 믿을 수 있게 만드는 데이터 전처리

이 노트북은 `notebooks/ch05_data_preprocessing.ipynb` 강의안을 실습하며 과제 답안을 작성한 자료입니다.

핵심 목표: 데이터를 무조건 깨끗하게 만드는 것이 아니라, **분석 결과를 신뢰할 수 있도록 데이터 상태를 확인하고 처리 기준을 남기는 것**입니다.

## 제출 정보

이름: 박성주 / GitHub ID: mikeypark / 작성일: 2026-10-14

## 학습 목표

원본 데이터를 복사해 전처리하고, 결측치, 중복, 이상값 처리 기준을 기록한다. 날짜형, 숫자형 변환과 파생 컬럼 생성 후 파일 간 키 관계를 재확인하고, 결과를 `data/processed/`와 보고서로 저장한다.

## 1. 패키지와 경로 설정

In [1]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == 'notebooks':
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR.parent if (CURRENT_DIR / '..').resolve().joinpath('requirements.txt').exists() else CURRENT_DIR

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('현재 실행 위치:', CURRENT_DIR)
print('프로젝트 루트:', PROJECT_ROOT)
print('원본 데이터 폴더:', RAW_DIR)
print('전처리 데이터 폴더:', PROCESSED_DIR)
print('보고서 폴더:', REPORT_DIR)

현재 실행 위치: C:\Users\uncle\Desktop\서울대 응용공학과\수업\2학기\대규모언어모델\git\llm-data-analysis-study\chapter05
프로젝트 루트: C:\Users\uncle\Desktop\서울대 응용공학과\수업\2학기\대규모언어모델\git\llm-data-analysis-study
원본 데이터 폴더: C:\Users\uncle\Desktop\서울대 응용공학과\수업\2학기\대규모언어모델\git\llm-data-analysis-study\data\raw
전처리 데이터 폴더: C:\Users\uncle\Desktop\서울대 응용공학과\수업\2학기\대규모언어모델\git\llm-data-analysis-study\data\processed
보고서 폴더: C:\Users\uncle\Desktop\서울대 응용공학과\수업\2학기\대규모언어모델\git\llm-data-analysis-study\reports


## 2. 원본 데이터 불러오기

In [2]:
customers = pd.read_csv(RAW_DIR / 'customers.csv')
products = pd.read_csv(RAW_DIR / 'products.csv')
orders = pd.read_csv(RAW_DIR / 'orders.csv')
order_items = pd.read_csv(RAW_DIR / 'order_items.csv')

print('원본 데이터 불러오기 완료')

원본 데이터 불러오기 완료


## 3. 전처리 전 데이터 크기 기록

In [3]:
raw_data = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

raw_shapes = pd.DataFrame({
    'dataset': list(raw_data.keys()),
    'rows': [df.shape[0] for df in raw_data.values()],
    'columns': [df.shape[1] for df in raw_data.values()],
})

display(raw_shapes)

,dataset,rows,columns
0,customers,150,6
1,products,100,4
2,orders,300,5
3,order_items,764,5


In [4]:
for name, df in raw_data.items():
    print(f'[{name}]')
    print('shape:', df.shape)
    print('columns:', list(df.columns))
    print()

[customers]
shape: (150, 6)
columns: ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']

[products]
shape: (100, 4)
columns: ['product_id', 'product_name', 'category', 'price']

[orders]
shape: (300, 5)
columns: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']

[order_items]
shape: (764, 5)
columns: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']



## 4. 원본을 복사해서 전처리 시작하기

In [5]:
customers_clean = customers.copy()
products_clean = products.copy()
orders_clean = orders.copy()
order_items_clean = order_items.copy()

clean_data = {
    'customers': customers_clean,
    'products': products_clean,
    'orders': orders_clean,
    'order_items': order_items_clean,
}

print('전처리용 복사본 생성 완료')

전처리용 복사본 생성 완료


## 5. 결측치 확인하기

In [6]:
for name, df in clean_data.items():
    print(f'\n[{name}] 결측치 개수')
    print(df.isna().sum())


[customers] 결측치 개수
customer_id    0
name           0
gender         0
age            0
city           0
signup_date    0
dtype: int64

[products] 결측치 개수
product_id      0
product_name    0
category        0
price           0
dtype: int64

[orders] 결측치 개수
order_id          0
customer_id       0
order_date        0
payment_method    0
order_status      0
dtype: int64

[order_items] 결측치 개수
order_item_id    0
order_id         0
product_id       0
quantity         0
unit_price       0
dtype: int64


In [7]:
def missing_summary(df):
    summary = pd.DataFrame({
        'missing_count': df.isna().sum(),
        'missing_ratio': (df.isna().mean() * 100).round(2),
    })
    return summary.sort_values('missing_count', ascending=False)

display(missing_summary(customers_clean))

,missing_count,missing_ratio
customer_id,0,0.0
name,0,0.0
gender,0,0.0
age,0,0.0
city,0,0.0
signup_date,0,0.0


**[결과 관찰]**

4개 데이터셋 모두 결측치 0건.

**[해석]**

결측치가 없더라도 처리 로직은 코드에 남겨둔다. 실데이터에서 나이 미입력, 도시 미기재 같은 경우가 생기며, 결측 비율이 높은 컬럼을 그대로 쓰면 집계가 편향된다.

## 6. 결측치 처리하기

In [8]:
if 'age' in customers_clean.columns:
    customers_clean['age'] = pd.to_numeric(customers_clean['age'], errors='coerce')
    age_median = customers_clean['age'].median()
    customers_clean['age'] = customers_clean['age'].fillna(age_median)
    print('age 중앙값:', age_median)

if 'city' in customers_clean.columns:
    customers_clean['city'] = customers_clean['city'].fillna('Unknown')

display(missing_summary(customers_clean))

age 중앙값: 40.0


,missing_count,missing_ratio
customer_id,0,0.0
name,0,0.0
gender,0,0.0
age,0,0.0
city,0,0.0
signup_date,0,0.0


**[결과 관찰]**

age 중앙값 40.0. 결측치 0건이라 처리 전후 missing_count 변화 없음.

**[해석]**

중앙값은 평균보다 이상값 영향을 덜 받는다. 결측이 많을 경우 해당 값 주위로 분포가 몰리는 효과가 생긴다.

## 7. 중복 확인하기

In [9]:
for name, df in clean_data.items():
    print(name, '전체 행 중복 수:', df.duplicated().sum())

customers 전체 행 중복 수: 0
products 전체 행 중복 수: 0
orders 전체 행 중복 수: 0
order_items 전체 행 중복 수: 0


In [10]:
key_checks = {
    'customers': ('customer_id', customers_clean),
    'products': ('product_id', products_clean),
    'orders': ('order_id', orders_clean),
    'order_items': ('order_item_id', order_items_clean),
}

for name, (key_col, df) in key_checks.items():
    if key_col in df.columns:
        print(name, key_col, '중복 수:', df[key_col].duplicated().sum())
    else:
        print(name, key_col, '컬럼 없음')

customers customer_id 중복 수: 0
products product_id 중복 수: 0
orders order_id 중복 수: 0
order_items order_item_id 중복 수: 0


In [11]:
customers_clean = customers_clean.drop_duplicates()
products_clean = products_clean.drop_duplicates()
orders_clean = orders_clean.drop_duplicates()
order_items_clean = order_items_clean.drop_duplicates()

print('완전 중복 행 제거 완료')

완전 중복 행 제거 완료


**[결과 관찰]**

전체 행 중복 0건. customer_id, product_id, order_id, order_item_id 중복 모두 0건.

**[해석]**

order_items의 order_id 반복은 정상(한 주문에 상품 여럿). order_item_id 중복은 오류다. 샘플은 0건이라 drop_duplicates 후에도 행 수 변화 없음.

## 8. 문자열 표기 정리하기

In [12]:
def strip_string_columns(df):
    result = df.copy()
    string_columns = result.select_dtypes(include=['object', 'str']).columns

    for col in string_columns:
        result[col] = result[col].where(
            result[col].isna(),
            result[col].astype(str).str.strip()
        )

    return result

customers_clean = strip_string_columns(customers_clean)
products_clean = strip_string_columns(products_clean)
orders_clean = strip_string_columns(orders_clean)
order_items_clean = strip_string_columns(order_items_clean)

print('문자열 앞뒤 공백 제거 완료')

문자열 앞뒤 공백 제거 완료


In [13]:
if 'city' in customers_clean.columns:
    display(customers_clean['city'].value_counts().head(10))

if 'order_status' in orders_clean.columns:
    display(orders_clean['order_status'].value_counts())

city
성남    21
광주    17
부산    16
대구    15
서울    15
울산    14
인천    14
대전    14
수원    13
고양    11
Name: count, dtype: int64

order_status
completed    184
cancelled     64
refunded      52
Name: count, dtype: int64

**[결과 관찰]**

도시: 성남(21), 광주(17), 부산(16) 순. 상태: completed(184), cancelled(64), refunded(52). 공백 제거 후에도 분포 변화 없음.

**[해석]**

공백이 숨어 있으면 value_counts()에서 'Seoul'과 ' Seoul'이 따로 집계된다. 변화가 없다면 원래 일관된 표기였다.

## 9. 주문 상태값 통일하기

In [14]:
status_map = {
    'complete': 'completed',
    'Complete': 'completed',
    'COMPLETED': 'completed',
    '완료': 'completed',
    'cancel': 'cancelled',
    'Cancel': 'cancelled',
    'CANCELLED': 'cancelled',
    '취소': 'cancelled',
    'refund': 'refunded',
    'Refund': 'refunded',
    'REFUNDED': 'refunded',
    '환불': 'refunded',
}

if 'order_status' in orders_clean.columns:
    orders_clean['order_status'] = orders_clean['order_status'].replace(status_map)
    display(orders_clean['order_status'].value_counts())

order_status
completed    184
cancelled     64
refunded      52
Name: count, dtype: int64

**[결과 관찰]**

통일 후에도 completed 184, cancelled 64, refunded 52로 변화 없음. 원본이 이미 소문자로 일관됨.

**[해석]**

replace()에 없는 값은 그대로 유지된다. 통일 후 value_counts()로 예상치 못한 표기가 남지 않았는지 확인한다.

## 10. 날짜 컬럼 변환하기

In [15]:
orders_clean['order_date'] = pd.to_datetime(orders_clean['order_date'], errors='coerce')
print('order_date 변환 실패:', orders_clean['order_date'].isna().sum())
print('주문 시작일:', orders_clean['order_date'].min())
print('주문 종료일:', orders_clean['order_date'].max())

orders_clean['order_month'] = orders_clean['order_date'].dt.to_period('M').astype(str)
orders_clean['order_dayofweek'] = orders_clean['order_date'].dt.day_name()

display(orders_clean[['order_date', 'order_month', 'order_dayofweek']].head())

order_date 변환 실패: 0
주문 시작일: 2025-07-09 00:00:00
주문 종료일: 2026-07-08 00:00:00


,order_date,order_month,order_dayofweek
0,2026-05-07,2026-05,Thursday
1,2025-07-23,2025-07,Wednesday
2,2025-11-19,2025-11,Wednesday
3,2026-01-30,2026-01,Friday
4,2025-12-21,2025-12,Sunday


In [16]:
if 'signup_date' in customers_clean.columns:
    customers_clean['signup_date'] = pd.to_datetime(customers_clean['signup_date'], errors='coerce')
    print('signup_date 변환 실패:', customers_clean['signup_date'].isna().sum())
    display(customers_clean[['customer_id', 'signup_date']].head())

signup_date 변환 실패: 0


,customer_id,signup_date
0,1,2024-06-19
1,2,2025-11-02
2,3,2024-06-12
3,4,2026-04-13
4,5,2024-09-13


**[결과 관찰]**

order_date 변환 실패 0건, 범위 2025-07-09 ~ 2026-07-08. signup_date 변환 실패 0건. order_month, order_dayofweek 파생 컬럼 생성 완료.

**[해석]**

errors='coerce'를 쓰면 변환 실패 시 NaT로 처리된다. 실패 건수가 0이 아니면 날짜 포맷이 혼재된 것이므로 원본을 확인해야 한다.

## 11. 숫자형 컬럼 변환하기

In [17]:
def to_number(series):
    return pd.to_numeric(
        series.astype(str).str.replace(',', '', regex=False),
        errors='coerce'
    )

products_clean['price'] = to_number(products_clean['price'])
order_items_clean['quantity'] = to_number(order_items_clean['quantity'])
order_items_clean['unit_price'] = to_number(order_items_clean['unit_price'])

print('price 변환 실패:', products_clean['price'].isna().sum())
print('quantity 변환 실패:', order_items_clean['quantity'].isna().sum())
print('unit_price 변환 실패:', order_items_clean['unit_price'].isna().sum())

price 변환 실패: 0
quantity 변환 실패: 0
unit_price 변환 실패: 0


## 12. 이상값 후보 확인하기

In [18]:
display(products_clean[['price']].describe())
display(order_items_clean[['quantity', 'unit_price']].describe())

,price
count,100.000000
mean,110040.000000
std,56433.910574
min,5000.000000
25%,65750.000000
50%,112000.000000
75%,161000.000000
max,200000.000000


,quantity,unit_price
count,764.000000,764.000000
mean,3.053665,108561.518325
std,1.410873,56996.770604
min,1.000000,5000.000000
25%,2.000000,62000.000000
50%,3.000000,111000.000000
75%,4.000000,161250.000000
max,5.000000,200000.000000


In [19]:
print('price <= 0:', len(products_clean[products_clean['price'] <= 0]))
print('quantity <= 0:', len(order_items_clean[order_items_clean['quantity'] <= 0]))
print('unit_price <= 0:', len(order_items_clean[order_items_clean['unit_price'] <= 0]))

price <= 0: 0
quantity <= 0: 0
unit_price <= 0: 0


## 13. 이상값 처리하기

실습용 기준:
- price가 0 이하인 상품은 분석 대상에서 제외
- quantity가 0 이하인 주문 상세는 분석 대상에서 제외
- unit_price가 0 이하인 주문 상세는 분석 대상에서 제외

In [20]:
products_clean = products_clean[products_clean['price'] > 0]
order_items_clean = order_items_clean[order_items_clean['quantity'] > 0]
order_items_clean = order_items_clean[order_items_clean['unit_price'] > 0]

print('이상값 처리 후 products:', products_clean.shape)
print('이상값 처리 후 order_items:', order_items_clean.shape)

이상값 처리 후 products: (100, 4)
이상값 처리 후 order_items: (764, 5)


**[결과 관찰]**

price > 0: 100행 유지. quantity > 0, unit_price > 0: 764행 유지.

**[해석]**

이상값 0건이라 필터 전후 행 수가 같다. 실데이터에서 price = 0은 이벤트 상품, 음수 quantity는 반품일 수 있어 무조건 제거 전에 의미를 확인해야 한다.

## 14. 파생 컬럼 만들기

`line_total = quantity × unit_price`

In [21]:
order_items_clean['line_total'] = (
    order_items_clean['quantity'] * order_items_clean['unit_price']
)

display(order_items_clean[['quantity', 'unit_price', 'line_total']].head())

,quantity,unit_price,line_total
0,3,102000,306000
1,5,25000,125000
2,3,142000,426000
3,3,193000,579000
4,4,189000,756000


In [22]:
print('전처리 후 주문 상세 금액 합계:', order_items_clean['line_total'].sum())

전처리 후 주문 상세 금액 합계: 255610000


## 15. 파일 간 관계 다시 확인하기

In [23]:
invalid_customers = orders_clean[
    ~orders_clean['customer_id'].isin(customers_clean['customer_id'])
]

invalid_orders = order_items_clean[
    ~order_items_clean['order_id'].isin(orders_clean['order_id'])
]

invalid_products = order_items_clean[
    ~order_items_clean['product_id'].isin(products_clean['product_id'])
]

relationship_checks = pd.DataFrame({
    'check': [
        'orders.customer_id exists in customers.customer_id',
        'order_items.order_id exists in orders.order_id',
        'order_items.product_id exists in products.product_id',
    ],
    'invalid_count': [
        len(invalid_customers),
        len(invalid_orders),
        len(invalid_products),
    ],
})

display(relationship_checks)

,check,invalid_count
0,orders.customer_id exists in customers.custome...,0
1,order_items.order_id exists in orders.order_id,0
2,order_items.product_id exists in products.prod...,0


**[결과 관찰]**

3가지 관계 모두 invalid_count = 0.

**[해석]**

price <= 0인 행이 없어 products 행 수가 유지됐고 관계도 그대로다. 행이 줄어드는 처리를 하면 이 단계를 반드시 재실행해야 한다.

## 16. 전처리 전후 비교하기

In [24]:
processed_data = {
    'customers': customers_clean,
    'products': products_clean,
    'orders': orders_clean,
    'order_items': order_items_clean,
}

processed_shapes = pd.DataFrame({
    'dataset': list(processed_data.keys()),
    'rows': [df.shape[0] for df in processed_data.values()],
    'columns': [df.shape[1] for df in processed_data.values()],
})

comparison = raw_shapes.merge(
    processed_shapes,
    on='dataset',
    suffixes=('_raw', '_processed')
)

display(comparison)

,dataset,rows_raw,columns_raw,rows_processed,columns_processed
0,customers,150,6,150,6
1,products,100,4,100,4
2,orders,300,5,300,7
3,order_items,764,5,764,6


**[결과 관찰]**

customers, products: 행/열 변화 없음. orders: 5열 → 7열(+order_month, +order_dayofweek). order_items: 5열 → 6열(+line_total).

**[해석]**

결측, 중복, 이상값이 없어 행이 줄지 않았다. 실데이터에서 행이 줄었다면 기준과 건수를 기록해야 이후 집계 차이를 추적할 수 있다.

## 17. 전처리 결과 저장하기

In [25]:
customers_clean.to_csv(PROCESSED_DIR / 'customers_clean.csv', index=False, encoding='utf-8-sig')
products_clean.to_csv(PROCESSED_DIR / 'products_clean.csv', index=False, encoding='utf-8-sig')
orders_clean.to_csv(PROCESSED_DIR / 'orders_clean.csv', index=False, encoding='utf-8-sig')
order_items_clean.to_csv(PROCESSED_DIR / 'order_items_clean.csv', index=False, encoding='utf-8-sig')

display(sorted(PROCESSED_DIR.glob('*_clean.csv')))

[WindowsPath('C:/Users/uncle/Desktop/서울대 응용공학과/수업/2학기/대규모언어모델/git/llm-data-analysis-study/data/processed/customers_clean.csv'),
 WindowsPath('C:/Users/uncle/Desktop/서울대 응용공학과/수업/2학기/대규모언어모델/git/llm-data-analysis-study/data/processed/order_items_clean.csv'),
 WindowsPath('C:/Users/uncle/Desktop/서울대 응용공학과/수업/2학기/대규모언어모델/git/llm-data-analysis-study/data/processed/orders_clean.csv'),
 WindowsPath('C:/Users/uncle/Desktop/서울대 응용공학과/수업/2학기/대규모언어모델/git/llm-data-analysis-study/data/processed/products_clean.csv')]

## 18. 전처리 요약 보고서 저장하기

In [26]:
summary_text = f'''# Chapter 5 데이터 전처리 요약

## 전처리 결과 파일

- customers_clean.csv
- products_clean.csv
- orders_clean.csv
- order_items_clean.csv

## 전처리 전후 데이터 크기

```text
{comparison.to_string(index=False)}
```

## 파일 간 관계 점검 결과

```text
{relationship_checks.to_string(index=False)}
```

## 주요 처리 내용

- 원본 데이터를 직접 수정하지 않고 복사본을 사용함
- 결측치 0건 확인 (샘플 데이터 기준), age 중앙값(40.0) 대체 로직 구성
- 전체 행 중복 0건, 주요 ID 중복 0건 확인
- 문자열 컬럼 앞뒤 공백 제거 (변화 없음)
- 주문 상태값 표기 통일 맵 적용 (변화 없음)
- order_date, signup_date 날짜형 변환 (실패 0건)
- order_month, order_dayofweek 파생 컬럼 생성
- price, quantity, unit_price 숫자형 변환 (실패 0건)
- 0 이하 이상값 0건 확인 및 처리 기준 적용
- line_total = quantity × unit_price 파생 컬럼 생성
- 파일 간 키 관계 재확인 (invalid 0건)

## 주의 사항

이 전처리 기준은 실습용 예시입니다. 실데이터에서는 결측치, 이상값 처리 전에
원본 시스템, 수집 과정, 업무 담당자 확인이 필요합니다.
'''

report_path = REPORT_DIR / 'ch05_preprocessing_summary.md'
report_path.write_text(summary_text, encoding='utf-8')

print('요약 보고서 저장 완료:', report_path)

요약 보고서 저장 완료: C:\Users\uncle\Desktop\서울대 응용공학과\수업\2학기\대규모언어모델\git\llm-data-analysis-study\reports\ch05_preprocessing_summary.md


## 19. LLM과 함께 전처리 코드를 검토하기

다음 프롬프트를 LLM에 입력했습니다.

```text
다음 전처리 코드가 안전한지 검토해

customers["age"] = customers["age"].fillna(customers["age"].mean())
customers = customers.drop_duplicates()
orders["order_date"] = pd.to_datetime(orders["order_date"])
order_items["line_total"] = order_items["quantity"] * order_items["unit_price"]

검토 기준:
- 원본 데이터를 직접 수정하는 문제가 있는지
- 결측치 처리 방식이 적절한지
- 날짜 변환 실패를 확인하는지
- 중복 제거 기준이 충분한지
- line_total 계산 전에 숫자형 변환이 필요한지
- 더 안전한 코드로 어떻게 수정할 수 있는지
```

**[LLM 제안 검토 결과]**

| 항목 | 판정 | 비고 |
|------|------|------|
| 원본 직접 수정 | ❌ | `customers = customers.copy()` 없이 원본에 직접 대입 |
| 결측치 처리 방식 | ⚠️ | 평균 대체는 이상값에 민감, 중앙값이 더 안전 |
| 날짜 변환 실패 확인 | ❌ | `errors='coerce'` 없음, 변환 실패 시 예외 발생 |
| 중복 제거 기준 | ⚠️ | 전체 행 기준 drop_duplicates는 키 중복과 구분 필요 |
| 숫자형 변환 선행 | ❌ | quantity, unit_price가 문자열이면 line_total 계산 오류 |

**[해석]**

LLM 코드는 작동 여부뿐 아니라 안전성 기준으로도 검증해야 한다. 특히 `.copy()` 없이 원본을 수정하면 같은 변수를 재사용할 때 의도치 않은 결과가 생긴다.

## 20. 실습 과제

아래 6가지 과제를 직접 해결합니다.

### 과제 1. 각 데이터셋의 결측치 비율을 하나의 표로 합쳐 보세요.

In [27]:
# 과제 1. 각 데이터셋의 결측치 비율을 하나의 표로 합쳐 보세요.
all_missing_parts = []

for name, df in raw_data.items():
    s = missing_summary(df).reset_index()
    s.columns = ['column', 'missing_count', 'missing_ratio']
    s['dataset'] = name
    all_missing_parts.append(s[['dataset', 'column', 'missing_count', 'missing_ratio']])

all_missing_df = pd.concat(all_missing_parts, ignore_index=True)

print('4개 데이터셋 통합 결측치 비율 표 (결측치 0건 컬럼 제외 시 모두 해당 없음)')
display(all_missing_df[all_missing_df['missing_count'] >= 0].head(10))

4개 데이터셋 통합 결측치 비율 표 (결측치 0건 컬럼 제외 시 모두 해당 없음)


,dataset,column,missing_count,missing_ratio
0,customers,customer_id,0,0.0
1,customers,name,0,0.0
2,customers,gender,0,0.0
3,customers,age,0,0.0
4,customers,city,0,0.0
5,customers,signup_date,0,0.0
6,products,product_id,0,0.0
7,products,product_name,0,0.0
8,products,category,0,0.0
9,products,price,0,0.0


**[결과 관찰]**

4개 데이터셋 전체 컬럼 결측치 비율 모두 0.0%.

**[해석]**

missing_summary(df).reset_index()로 컬럼 이름을 행으로 꺼낸 뒤 dataset 컬럼을 추가해 pd.concat으로 합쳤다.

### 과제 2. customers_clean에서 나이가 18세 미만이거나 100세 초과인 값이 있는지 확인하세요.

In [28]:
# 과제 2. customers_clean에서 나이가 18세 미만이거나 100세 초과인 값이 있는지 확인하세요.
age_too_young = customers_clean[customers_clean['age'] < 18]
age_too_old = customers_clean[customers_clean['age'] > 100]

print(f'나이 18세 미만: {len(age_too_young)}건')
print(f'나이 100세 초과: {len(age_too_old)}건')

if len(age_too_young) == 0 and len(age_too_old) == 0:
    print(f'이상 연령 없음 - 전처리 후 고객 나이 범위: {customers_clean["age"].min()}세 ~ {customers_clean["age"].max()}세')
else:
    print('이상 연령 발견 - 처리 기준 재검토 필요')
    display(pd.concat([age_too_young, age_too_old])[['customer_id', 'age', 'city']])

나이 18세 미만: 0건
나이 100세 초과: 0건
이상 연령 없음 - 전처리 후 고객 나이 범위: 19세 ~ 69세


**[결과 관찰]**

18세 미만 0건, 100세 초과 0건. 전처리 후 나이 범위: 19~69세.

**[해석]**

실데이터에서 age = 0이나 999 같은 값은 입력 오류 가능성이 높다. 단순 삭제 시 해당 고객의 모든 주문이 분석에서 빠지므로 원본 확인 후 처리한다.

### 과제 3. orders_clean에서 월별 주문 건수를 계산하세요.

In [29]:
# 과제 3. orders_clean에서 월별 주문 건수를 계산하세요.
monthly_order_count = (
    orders_clean
    .groupby('order_month', as_index=False)
    .agg(order_count=('order_id', 'count'))
    .sort_values('order_month')
    .reset_index(drop=True)
)

display(monthly_order_count)
print(f'월별 합계: {monthly_order_count["order_count"].sum()} (전체 주문 수와 일치 여부: {monthly_order_count["order_count"].sum() == len(orders_clean)})')

,order_month,order_count
0,2025-07,15
1,2025-08,24
2,2025-09,23
3,2025-10,39
4,2025-11,23
5,2025-12,26
6,2026-01,29
7,2026-02,29
8,2026-03,21
9,2026-04,31


월별 합계: 300 (전체 주문 수와 일치 여부: True)


**[결과 관찰]**

총 13개월(2025-07 ~ 2026-07). 최다 주문: 2025-10(39건). 최소: 2026-07(4건). 전체 합계 300건 일치.

**[해석]**

모든 주문 상태 포함 기준이다. 매출 분석에는 completed 필터링이 필요하다. 2026-07은 4건뿐이라 트렌드 해석에 주의한다.

### 과제 4. order_items_clean에서 line_total이 큰 순서대로 상위 10개를 확인하세요.

In [30]:
# 과제 4. order_items_clean에서 line_total이 큰 순서대로 상위 10개를 확인하세요.
top10_line_total = (
    order_items_clean
    .nlargest(10, 'line_total')
    [['order_id', 'product_id', 'quantity', 'unit_price', 'line_total']]
    .reset_index(drop=True)
)

display(top10_line_total)

,order_id,product_id,quantity,unit_price,line_total
0,10,99,5,200000,1000000
1,39,99,5,200000,1000000
2,76,99,5,200000,1000000
3,35,70,5,198000,990000
4,190,70,5,198000,990000
5,21,58,5,197000,985000
6,25,58,5,197000,985000
7,51,58,5,197000,985000
8,152,58,5,197000,985000
9,221,58,5,197000,985000


**[결과 관찰]**

1위: product_id 99, 수량 5개 × 200,000원 = 1,000,000원. 상위 10건 모두 수량 5(최대)에 단가가 높은 상품.

**[해석]**

전체 주문 상태 포함 기준이다. completed 주문만 보려면 orders_clean과 병합 후 필터링이 필요하다.

### 과제 5. 전처리 전후 행 수가 줄어든 데이터셋이 있는지 설명해 보세요.

**답변:**

이번 샘플 데이터에서는 전처리 전후 행 수가 줄어든 데이터셋이 없다.

결측치 0건, 중복 0건, 이상값 0건이라 필터링 후에도 행 수가 유지됐다. 대신 열 수가 변경됐다. orders는 5열 → 7열(+order_month, +order_dayofweek), order_items는 5열 → 6열(+line_total).

실데이터에서 행이 줄어들 경우, 어떤 기준으로 몇 행이 제거됐는지 기록해 두어야 이후 집계 차이를 추적할 수 있다.

### 과제 6. LLM에게 결측치 처리 기준을 검토해 달라는 프롬프트를 직접 작성해 보세요.

**작성한 프롬프트:**

```text
다음 결측치 처리 코드의 기준이 분석 목적에 적합한지 검토해

데이터셋 정보:
- customers.csv: 고객 정보 (150행, 6컬럼)
  - age: int, 결측치 0건 (샘플), 범위 19~69
  - city: str, 결측치 0건 (샘플), 10개 도시

처리 코드:
customers_clean['age'] = customers_clean['age'].fillna(customers_clean['age'].median())
customers_clean['city'] = customers_clean['city'].fillna('Unknown')

분석 목적: 나이, 도시별 고객 구매 패턴 분석

검토 기준:
1. age 중앙값 대체가 나이 분포 분석에 편향을 일으키는지
2. city 'Unknown' 처리 시 도시별 집계에서 이 그룹을 어떻게 다뤄야 하는지
3. 중앙값 대체보다 더 적합한 방법이 있는지
4. 결측치 비율이 높을 경우(30% 이상) 처리 방식을 달리해야 하는지
5. 개선이 필요하다면 수정 코드도 제안
```

**[해석]**

컬럼명, 타입, 결측치 개수, 분석 목적만 제공해 개인정보 유출 없이 LLM 검토를 받을 수 있다. 검토 기준을 명시하면 답변이 더 구체적이다.

## Chapter 05 최종 인사이트

### 가장 의미 있다고 생각한 결과 2가지

**결측, 중복, 이상값 모두 0건**: 샘플이라 깨끗하지만, 각 확인 시점의 순서—이상값 확인 후 필터링, 필터링 후 관계 재확인—가 실데이터 작업 순서와 같다.

**orders 열이 5 → 7로 늘어남**: order_month, order_dayofweek 파생 컬럼은 실제 분석에 바로 쓸 수 있는 형태다. 전처리는 단순 정리가 아니라 분석 목적에 맞는 재가공이다.

### 추가로 확인하고 싶은 질문

2025-10 주문이 39건으로 가장 많은 이유는 무엇인가(시즌, 프로모션). 요일별 주문 패턴이 있는가(주중 vs 주말). signup_date와 첫 주문일 간격이 구매 패턴에 영향을 주는가.

### 현재 결과의 한계

샘플 데이터라 전처리 효과를 직접 경험하기 어렵다. line_total은 주문 상태 미반영 기준이다.